### Setup and Data Loading ###

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

# Load data and prep splits (exactly as before to ensure consistency)
df = pd.read_csv("../data/diabetes_cleaned.csv")
X = df.drop(columns=["Outcome"])
y = df["Outcome"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Data loaded and scaled successfully.")

Data loaded and scaled successfully.


### Part F - Naïve Bayes Experiment ###
For Gaussian Naïve Bayes, we will experiment with the var_smoothing hyperparameter by testing the default value, a smaller value, and a larger value

In [2]:
from sklearn.naive_bayes import GaussianNB

# Define variance smoothing parameters to test
smoothing_params = {
    "Default (1e-9)": 1e-9,
    "Smaller (1e-11)": 1e-11,
    "Larger (1e-1)": 1e-1
}

print("Naïve Bayes - Variance Smoothing Experiment:")
print("-" * 45)

for label, var_smoothing in smoothing_params.items():
    nb = GaussianNB(var_smoothing=var_smoothing)
    nb.fit(X_train_scaled, y_train)
    
    y_pred = nb.predict(X_test_scaled)
    acc = accuracy_score(y_test, y_pred)
    
    print(f"{label.ljust(18)} : Accuracy = {acc:.4f}")

Naïve Bayes - Variance Smoothing Experiment:
---------------------------------------------
Default (1e-9)     : Accuracy = 0.7273
Smaller (1e-11)    : Accuracy = 0.7273
Larger (1e-1)      : Accuracy = 0.7208


### Part G - Decision Tree Experiment ###
Building a shallow tree, a medium-depth tree, and a deep tree to demonstrate how depth affects training vs. testing performance (overfitting).

In [4]:
from sklearn.tree import DecisionTreeClassifier

# Define tree depths to test
tree_depths = {
    "Shallow Tree (max_depth=3)": 3,
    "Medium Tree (max_depth=7)": 7,
    "Deep Tree (max_depth=None)": None
}

print("Decision Tree - Depth and Overfitting Experiment:")
print("-" * 55)

for label, depth in tree_depths.items():
    # Initialize and train
    dt = DecisionTreeClassifier(max_depth=depth, random_state=42)
    dt.fit(X_train_scaled, y_train) # Trees don't strictly require scaling, but it's fine here
    
    # Evaluate on both Train and Test to check for overfitting
    train_acc = accuracy_score(y_train, dt.predict(X_train_scaled))
    test_acc = accuracy_score(y_test, dt.predict(X_test_scaled))
    
    print(f"{label}:")
    print(f"  -> Training Accuracy : {train_acc:.4f}")
    print(f"  -> Testing Accuracy  : {test_acc:.4f}\n")

Decision Tree - Depth and Overfitting Experiment:
-------------------------------------------------------
Shallow Tree (max_depth=3):
  -> Training Accuracy : 0.8811
  -> Testing Accuracy  : 0.8506

Medium Tree (max_depth=7):
  -> Training Accuracy : 0.9723
  -> Testing Accuracy  : 0.8506

Deep Tree (max_depth=None):
  -> Training Accuracy : 1.0000
  -> Testing Accuracy  : 0.8117



### Observations: Naïve Bayes & Decision Tree Tuning

#### 1. Naïve Bayes (Variance Smoothing)
* **Default vs. Smaller (1e-9 vs. 1e-11):** Both produced identical test accuracy (**72.73%**), indicating that smaller variance adjustments do not alter class boundary estimates for this dataset.
* **Larger Smoothing (1e-1):** Slightly degraded test accuracy to **72.08%**, as over-smoothing smooths feature variances too aggressively, dampening class separation capability.

#### 2. Decision Tree (Depth & Overfitting)
* **Shallow Tree (`max_depth=3`):** Achieved **85.06%** test accuracy with an 88.11% training accuracy. The small gap indicates strong generalization with minimal overfitting.
* **Medium Tree (`max_depth=7`):** Reached 97.23% training accuracy but maintained the same test accuracy (**85.06%**).
* **Deep Tree (`max_depth=None`):** Suffered from classic **overfitting**. It achieved a perfect **100% training accuracy** (memorizing noise), but test accuracy dropped significantly to **81.17%**.
* **Takeaway:** Restricting tree depth is essential to prevent overfitting on medical tabular data.